# PARITY — the LLM-authored corpus, rewritten on a T4

The first LLM corpus (`latent/data/corpus_llm.jsonl`, qwen2.5:7b on a laptop) drops fields: the place is
missing from about a third of its sentences, and the latent gate error rises from 0.13 to 0.50 (STATE.md).
This notebook rewrites all 5000 with `latent.llm_corpus anchor`: the model tags the words that carry each
field inside its own sentence, and code checks every sentence (all five tagged, the place words name the
right place, each tag nearest its own option, no template copying), rewriting failures with the problem
stated. Then `compare` measures all three corpora.

**Run it top to bottom.** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
**Approve the Google Drive prompt** (cell 2): work in progress is kept there, so if Colab disconnects,
just `Run all` again and it resumes where it stopped.

Expect roughly 2–4 h, most of it in cell 7. Keep the tab open. Cell 6 is a 24-sentence calibration: if
too few sentences pass, it stops the run on purpose — send its output back instead of continuing.

At the end a zip downloads (and a copy is left in Drive under `parity_corpus/`). Bring it back.

## 0 — Confirm the GPU

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", out or "none")
assert out, "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again."

## 1 — Clean clone of the repo

In [ ]:
import os, shutil, subprocess
REPO, URL = "/content/cgvr", "https://github.com/chhavi-pareek/cgvr"
os.chdir("/content")                      # a re-run stands inside REPO; never delete the cwd
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--depth", "1", URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 2 — Keep the work in progress on Drive (so a disconnect loses nothing)

In [ ]:
import pathlib
from google.colab import drive
drive.mount("/content/drive")
KEEP = pathlib.Path("/content/drive/MyDrive/parity_corpus")
KEEP.mkdir(parents=True, exist_ok=True)
part = pathlib.Path(REPO) / "latent/data/corpus_llm_anchored.part.jsonl"
part.unlink(missing_ok=True)
part.symlink_to(KEEP / "corpus_llm_anchored.part.jsonl")
done = sum(1 for _ in open(part)) if part.exists() else 0
print(f"attempts already on Drive: {done}", "(resuming)" if done else "(fresh start)")

## 3 — Install Ollama and start it (4 requests in parallel)

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /content/ollama_install.log 2>&1; tail -2 /content/ollama_install.log

In [ ]:
import time, urllib.request
env = dict(os.environ, OLLAMA_NUM_PARALLEL="4", OLLAMA_KEEP_ALIVE="-1", OLLAMA_HOST="127.0.0.1:11434")
srv = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"),
                       stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama did not start; see /content/ollama.log")

## 4 — Pull the author model and check it sits entirely on the GPU

`qwen2.5:14b` (4-bit, ~9 GB): the 7B used for the first pass could not hold the tagged format (STATE.md).
If `ollama ps` shows anything other than `100% GPU`, it is partly on the CPU and will be ~10x slower.

In [ ]:
MODEL = "qwen2.5:14b"
os.environ["PARITY_LLM"] = MODEL
!ollama pull {MODEL} 2>&1 | tail -1
!ollama run {MODEL} "Reply with one word: ready" 2>/dev/null
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
if "100% GPU" not in ps:
    print("!! The model is not fully on the GPU. It will run, but slowly.")

## 5 — Python packages and the unit tests for the checks

In [ ]:
!pip install -q sentence-transformers scikit-learn 2>&1 | tail -1
!python -m pytest -q tests/test_llm_corpus.py 2>&1 | tail -2

## 6 — Calibration: 24 sentences, in a scratch folder

Prints the acceptance rate, the most common problems and the accepted sentences. It **stops the notebook**
if fewer than half pass within the allowed attempts: that means the prompt or the checks need changing,
and the full run would waste hours. Send this cell's output back in that case.

In [ ]:
import json, collections
import latent.llm_corpus as L
L.MODEL = MODEL
scratch = pathlib.Path("/content/calib"); shutil.rmtree(scratch, ignore_errors=True); scratch.mkdir()
L.PART, L.ANCHORED = scratch / "part.jsonl", scratch / "anchored.jsonl"
_finish, L.finish = L.finish, (lambda *a, **k: None)
t = time.perf_counter()
L.anchor(n=24, workers=4)
L.finish = _finish
rows = [json.loads(l) for l in open(L.PART)]
ok = {r["k"] for r in rows if r["ok"]}
print(f"\n{len(ok)}/24 accepted, {len(rows)} attempts, {(time.perf_counter() - t) / 60:.1f} min")
print(collections.Counter(p.split('"')[0][:40] for r in rows for p in r["problems"]).most_common(6))
for r in rows:
    if r["ok"]:
        print("  ", r["text"])
assert len(ok) >= 12, "Too few sentences pass: stop here and send this output back."

## 7 — The full run (the long one)

5000 combinations, 4 requests in parallel, up to 4 attempts each. It prints progress every 75 combinations.
If Colab disconnects, `Run all` again: cell 2 reconnects the Drive file and this resumes.

In [ ]:
!cd {REPO} && python -u -m latent.llm_corpus anchor 5000 4 2>&1 | grep -v "Loading weights" | tee /content/anchor.log | tail -40

## 8 — Compare template, first-pass LLM and rewritten LLM corpora

Diversity, near-duplicates, how well each field can be read back from the embedding, the salience prior,
and the latent gate (nested AE, k = 16, 3 seeds) on each.

In [ ]:
!cd {REPO} && python -u -m latent.llm_corpus compare 2>&1 | grep -v "Loading weights" | tee /content/compare.log

## 9 — Package the results (downloads a zip; a copy stays in Drive)

In [ ]:
import zipfile
z = pathlib.Path("/content/parity_corpus_results.zip")
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f:
    for p in ["latent/data/corpus_llm_anchored.jsonl", "latent/data/targets_llm_anchored.npz",
              "latent/data/corpus_llm_anchored.part.jsonl"]:
        f.write(pathlib.Path(REPO) / p, p)
    for p in ["/content/anchor.log", "/content/compare.log", "/content/calib/part.jsonl"]:
        if os.path.exists(p):
            f.write(p, "logs/" + os.path.basename(p) if "calib" not in p else "logs/calib_part.jsonl")
shutil.copy(z, KEEP / z.name)
print(z, f"{z.stat().st_size / 1e6:.1f} MB; copy in Drive: parity_corpus/{z.name}")
from google.colab import files
files.download(str(z))